# FOMC预定会议来源资格
从上一份已发布纪要读取下一场日期，发布时间从官方日历绑定。保留计划/实际差异和单独取消事件。日程来源资格不等于收益信息。0拟合、0回测。

In [1]:
import os,json,qlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/fomc_schedule_source_20261014'
qlib.init(provider_uri=str(root/'data/qlib/soxl'),region='us',kernels=1)
from src.workflow.fomc_schedule_source import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
summary

{'experiment': 'fomc_schedule_source_20261014',
 'plan_sha256': '0dea533e760685c2d104bb2a417b116ffc380710d5bf0e9403e06c34298a1cde',
 'decision': 'date_chain_collected_needs_event_timing_and_information_test',
 'objective_achieved': False,
 'new_fits': 0,
 'new_backtests': 0,
 'minutes_documents': 65,
 'qualified_past_date_events': 61,
 'status_counts': {'prior_public_date_evidence': 63,
  'future_schedule_no_market_outcome': 1,
  'cancelled_publicly_before_entry': 1},
 'original_first_receipt_certified': False,
 'current_calendar_used_as_advance_signal': False,
 'announcement_clock_certified_per_event': False,
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/soxl',
 'region': 'us',
 'as_of': '2026-10-14',
 'market_as_of': '2026-10-09',
 'qlib_docs': 'https://qlib.readthedocs.io/en/latest/component/recorder.html',
 'recorder_id': '61b1a4712c46489fa4a576cbaa85e9c2',
 'experiment_id': '305024794321636945',
 'recorder_uri': '/Users/rziiiii/dev/workspace/research/fomc_schedule_sourc

# FOMC日程：建立会前公开日期证据链

0拟合、0回测。冻结4份官方日历/历史索引、65份官方会议纪要以及1份取消说明PDF，共70份原始文档，保存URL、获取UTC、字节数和SHA256。完整原始文档与Qlib日历均进入NAS Recorder，本地为缓存。

方法不是把今天的完整会议日历直接当作历史交易输入。每份纪要正文中的“下一场会议”给出计划日期，官方索引标记该纪要的发布日期；只有发布日期日末（按美东解释）早于下一场会议结束日前一原生交易日09:46，才通过日期可用性检查。当前实际会议记录只用于核对一致性，不替代预告来源。

修复一项实际解析缺陷：2020年7月纪要在下一会议日期数字之间包含U+200D不可见格式字符。保留原始字节，解析时移除该字符；没有修改会议日期，也没有重新下载或用市场结果选择解析方式。初次漏读已修复，最终无未解析日期。

## 覆盖与例外

- 2019年至2026-10-09，共61场过去会议有会前公开日期证据：2019年8场、2020年7场、2021–2025各8场、2026年6场。
- 另保留2018年的2场已核对预告，未计入2019起研究窗口。
- 2026-10-28是已预告但尚无本次行情结果的未来会议，不能计为历史观察。
- 原定2020-03-17至18会议已被替代。3月15日主席公开电话记者会明确说明当天会议替代随后周二、周三的会议；本卡保守采用3月16日日末为取消信息可用边界，早于3月17日拟议入场。保留原计划及取消记录，不把3月15日临时会议伪装成提前预定事件。不要把同期Board贴现率会议的取消公告当作FOMC证据。

来源：[官方会议日历](https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm)、[2020历史索引](https://www.federalreserve.gov/monetarypolicy/fomchistorical2020.htm)、[取消说明：主席3月15日记者会第5页](https://www.federalreserve.gov/mediacenter/files/FOMCpresconf20200315.pdf)。

机制依据：[纽约联储2018年更新研究](https://libertystreeteconomics.newyorkfed.org/2018/11/the-pre-fomc-announcement-drift-more-recent-evidence/)讨论过预定会议前的股票收益，并指出效应随记者会安排及交易提前而变化；它没有证明当前半导体或SOXL存在稳定扣费超额。2019年起记者会制度变化是一个事先可说明的研究边界，不是根据本项目收益挑起点。

决定：**date_chain_collected_needs_event_timing_and_information_test**。这里只完成日期证据链，尚未逐场认证声明发布时钟、原始实时首收或全文修订版本，不能直接称作可交易完整事件数据。日期级官方发布日期与保守日末假设不等于历史实盘接收证明。

下一步固定一个会前持有窗口，逐场检查执行时点早于声明，并用现有真实分钟数据检验SOXL/SOXX差额及真实Tiered成本；保留取消记录、所有费用情景及压力日。禁止把公告后的跳涨算入会前漂移，也不按结果改日期或挑会议。只有完整净收益/风险/稳定性和独立前向门都通过，才可能达到总目标。

验证：日期跨月、出版日不同于会议日、不可见字符及无预告语句的合同测试通过；Notebook执行、65文档解析与NAS原始对象回读通过。[完整日期证据表](schedule_evidence.csv)、[Recorder清单](verification_manifest.json)。稳定净SOXX超额目标未完成；没有新模型、订单或SIM改动。
